# 03 · Figure 3: detector score profiles

Figure 3 plots the detector score $s_\ell$ over the eligible layers of four models:
the mean over the 1,000 edited checkpoints with the middle 50% and the full range shaded, the unedited control,
and the edited layer $\ell^*$. The notebook redraws the figure exactly as it appears in the paper
(`outputs/gram_profiles.pdf`, the file used as `img/detection/gram_profiles.pdf`), checks the statements of its caption,
and draws the same profiles for all 17 models as a supplementary figure (`outputs/gram_profiles_all_models.pdf`).

**Input:** `data/classic-layer-profile-statistics.csv`, one row per model and eligible layer with the unedited score
(`baseline_score`) and the distribution of the edited scores (`edited_mean`, `edited_min`, `edited_q25`, `edited_q75`, `edited_max`).

In [ ]:
import pandas as pd
from paper_common import (MODELS, EXCLUDED, OUT, read_csv, base_model_results, inputs_used,
                          Checks, latex_rows_equal, diff_rows)

pd.set_option("display.precision", 2)
checks = Checks()

import matplotlib.pyplot as plt

df = base_model_results().set_index("key")
prof = read_csv("data/classic-layer-profile-statistics.csv")
prof = prof[~prof.model.isin(EXCLUDED)]
checks.value("models with a profile", "Fig. 3", prof.model.nunique(), 17)
checks.value("edited checkpoints per model", "Fig. 3 caption", prof.n_edited_profiles.min(), 1000)
prof.head()

## Statements in the caption

In [ ]:
def peak(key, column):
    p = prof[prof.model == key]
    return int(p.loc[p[column].idxmax(), "layer"])

checks.same("Qwen3-8B: the edit creates the highest mean score at l*", "Fig. 3 caption",
            peak("qwen3-8b", "edited_mean"), int(df.loc["qwen3-8b", "target"]))
checks.value("GPT-2 Medium: natural (unedited) peak at layer 14", "Fig. 3 caption", peak("gpt2-medium", "baseline_score"), 14)
checks.same("Mistral-7B v0.3: the control already peaks at l*", "Fig. 3 caption",
            peak("mistral-7b-v0.3", "baseline_score"), int(df.loc["mistral-7b-v0.3", "target"]))
checks.value("Olmo 3 7B: natural peak at layer 4", "Fig. 3 caption", peak("olmo-3-1025-7b", "baseline_score"), 4)
checks.printed("Olmo 3 7B: share of edits localized (almost none)", "Fig. 3 caption", df.loc["olmo-3-1025-7b", "exact_pct"], "0.3")

## Figure 3

The panel notes are computed from the data (exact rate, and whether the control peaks at $\ell^*$).

In [ ]:
EDIT, EDIT_IQR, EDIT_RANGE, CTRL, MUTED, GRID = "#2a78d6", "#a9c9f0", "#dbe8f8", "#52514e", "#8a8984", "#e6e5e1"
STYLE = {"font.size": 7, "axes.titlesize": 7.5, "axes.labelsize": 7, "xtick.labelsize": 6.5, "ytick.labelsize": 6.5,
         "legend.fontsize": 7, "axes.spines.top": False, "axes.spines.right": False, "axes.linewidth": 0.6,
         "xtick.major.width": 0.6, "ytick.major.width": 0.6, "pdf.fonttype": 42}
PLAIN = lambda name: name.replace("~", " ")

def note(key):
    r = df.loc[key]
    text = "exact 100%" if r.exact_pct == 100 else f"exact {r.exact_pct:.1f}%"
    return text + (", control peaks at $\\ell^*$" if r.control_flags else "")

def panel(ax, key, title=None):
    p = prof[prof.model == key].sort_values("layer")
    x, t = p.layer, int(df.loc[key, "target"])
    ax.fill_between(x, p.edited_min, p.edited_max, color=EDIT_RANGE, lw=0, label="edited: min–max")
    ax.fill_between(x, p.edited_q25, p.edited_q75, color=EDIT_IQR, lw=0, label="edited: middle 50%")
    ax.plot(x, p.edited_mean, color=EDIT, lw=1.2, label="edited: mean (1,000 edits)")
    ax.plot(x, p.baseline_score, color=CTRL, lw=1.0, ls=(0, (3, 1.5)), label="unedited control")
    ax.axvline(t, color=MUTED, lw=0.7, ls=":", zorder=0)
    ax.annotate(f"$\\ell^*={t}$", xy=(t, 0.97), xycoords=("data", "axes fraction"), xytext=(3, 0), textcoords="offset points",
                ha="left", va="top", fontsize=6.5, color=CTRL, bbox=dict(boxstyle="square,pad=0.1", fc="white", ec="none", alpha=0.85))
    ax.set_title(title or PLAIN(df.loc[key, "name"]), loc="left", pad=11, fontweight="bold")
    ax.text(0, 1.015, note(key), transform=ax.transAxes, fontsize=6.5, color=CTRL, va="bottom", ha="left")
    ax.set_xlabel("Layer $\\ell$")
    ax.grid(axis="y", color=GRID, lw=0.5)
    ax.set_axisbelow(True)
    ax.set_ylim(bottom=0)

def legend(fig, ax):
    h, l = ax.get_legend_handles_labels()
    order = [2, 1, 0, 3]
    fig.legend([h[i] for i in order], [l[i] for i in order], loc="outside lower center", ncol=4, frameon=False)

PANELS = ["qwen3-8b", "gpt2-medium", "mistral-7b-v0.3", "olmo-3-1025-7b"]
with plt.rc_context(STYLE):
    fig, axes = plt.subplots(1, 4, figsize=(7.0, 1.75), constrained_layout=True)
    for ax, key in zip(axes, PANELS):
        panel(ax, key)
    axes[0].set_ylabel("Score $s_\\ell$")
    legend(fig, axes[0])
    fig.savefig(OUT / "gram_profiles.pdf", bbox_inches="tight", pad_inches=0.01)
    fig.savefig(OUT / "gram_profiles.png", dpi=200, bbox_inches="tight", pad_inches=0.01)
    plt.show()

## Supplementary: score profiles of all 17 models

The same panels for every model, in the order of Table 3.

In [ ]:
with plt.rc_context(STYLE):
    fig, axes = plt.subplots(5, 4, figsize=(7.0, 9.0), constrained_layout=True)
    for ax, key in zip(axes.flat, MODELS.key):
        panel(ax, key)
    for ax in axes.flat[len(MODELS):]:
        ax.set_visible(False)
    for row in axes[:, 0]:
        row.set_ylabel("Score $s_\\ell$")
    legend(fig, axes.flat[0])
    fig.savefig(OUT / "gram_profiles_all_models.pdf", bbox_inches="tight", pad_inches=0.01)
    fig.savefig(OUT / "gram_profiles_all_models.png", dpi=200, bbox_inches="tight", pad_inches=0.01)
    plt.show()

## Summary of checks

In [ ]:
checks.table().to_csv(OUT / "checks_03_figure.csv", index=False)
checks.assert_all()
checks.table()

In [ ]:
inputs_used()